# 第六课：RAG与AI智能体 —— 让AI开口说「真话」

## 学习目标
- 理解 RAG（检索增强生成）的核心原理
- 用代码搭建一个最简单的 RAG 问答系统
- 理解 AI 智能体的概念：工具使用、规划、记忆
- 对比纯模型回答和 RAG 回答的差异

> RAG 是减少 AI 幻觉、让回答有据可查的最实用技术之一。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：搭建最简单的 RAG 问答

### 活动目标
用代码搭建一个微型 RAG 系统：先准备几段「知识」，然后让 AI 在回答前先检索相关知识。

RAG 的核心思想很简单：**回答问题前，先查资料**。

RAG 三部曲：
1. **检索（Retrieve）**：从知识库中找到相关内容
2. **增强（Augment）**：把找到的内容加入到提示词中
3. **生成（Generate）**：AI 基于查到的资料生成回答

In [ ]:
# 活动一：微型 RAG 系统

# 第一步：准备知识库（几个「文档」）
knowledge_base = [
    {'title': '公司简介', 'content': 'ABC科技成立于2020年，总部位于深圳。'
     '公司专注于AI教育产品，主要产品包括AI学习助手和智能题库系统。'
     '2024年营收达到5000万元，员工数量约200人。'},
    {'title': '产品信息', 'content': 'AI学习助手是ABC科技的旗舰产品，'
     '支持数学、英语、编程等20+学科的智能辅导。'
     '产品采用GPT-4底层模型，月活跃用户超过100万。'
     '定价：个人版99元/月，企业版1999元/年。'},
    {'title': '联系方式', 'content': '客服电话：400-123-4567。'
     '商务合作请联系：bd@abc-tech.com。'
     '公司地址：深圳市南山区科技园路88号创新大厦15层。'
     '工作时间：周一至周五 9:00-18:00。'},
    {'title': '融资历史', 'content': 'ABC科技于2021年完成天使轮融资500万元，'
     '2022年完成A轮融资3000万元，2024年完成B轮融资1.5亿元。'
     'B轮由红杉资本领投，估值达到15亿元。'}
]

# 第二步：简单的关键词检索（检索）
import re

# 注意：中文句子里没有空格，不能像英文那样直接 query.split()——
# 否则整句话会被当成「一个词」，一篇文档都检索不到。
# 这里用最朴素的「二字滑窗」切词：「客服电话」→ 客服 / 服电 / 电话。
STOP_WORDS = {'的', '了', '是', '我', '你', '他', '们', '多少', '什么', '请问', '他们'}


def tokenize(text):
    """把一句话切成可用于匹配的词：英文数字按空格切，中文用二字滑窗"""
    tokens = re.findall(r'[a-zA-Z0-9]+', text.lower())
    for run in re.findall(r'[\u4e00-\u9fff]+', text):
        if len(run) == 1:
            tokens.append(run)
        else:
            tokens += [run[i:i + 2] for i in range(len(run) - 1)]
    return [t for t in tokens if t not in STOP_WORDS]


def simple_search(query, kb, top_k=2):
    """最简单的关键词匹配检索：数一数 query 的词在文档里命中了几个"""
    results = []
    for doc in kb:
        content = doc['content'].lower()
        score = sum(1 for word in tokenize(query) if word in content)
        if score > 0:
            results.append((score, doc))
    # 按匹配度排序，只取最相关的 top_k 篇
    # （不截断的话整个知识库都会被「检索」出来，检索就失去意义了）
    results.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in results[:top_k]]

print('知识库已就绪！共', len(knowledge_base), '篇文档。')

### 对比实验：不带 RAG vs 带 RAG

In [ ]:
# 对比实验：不带 RAG vs 带 RAG

question = 'ABC科技的客服电话是多少？他们的B轮融资是谁领投的？'

# 方式一：不带 RAG（纯模型回答）
print('=== 方式一：不带 RAG（纯模型回答）===')
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':question}],
    temperature=0)
print(r1.choices[0].message.content)

# 方式二：带 RAG
print('\n=== 方式二：带 RAG（检索后回答）===')
# 检索相关文档
relevant_docs = simple_search(question, knowledge_base)
print(f'检索到 {len(relevant_docs)} 篇相关文档：')
for doc in relevant_docs:
    print(f'  - {doc["title"]}')

# 将检索到的文档拼接成上下文
context = '\n\n'.join([f'【{doc["title"]}】\n{doc["content"]}' for doc in relevant_docs])

# 带上下文的 prompt
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'请只根据以下提供的资料回答问题。如果资料中没有相关信息，请明确说不知道。'},
        {'role':'user','content':f'参考资料：\n{context}\n\n问题：{question}'}
    ],
    temperature=0)
print(r2.choices[0].message.content)

print('\n对比两种方式的回答：RAG 的回答是否更准确、更有依据？')

### 讨论
- RAG 的回答和纯模型回答有什么不同？
- 当知识库中没有相关信息时，RAG 方式的 AI 会怎么说？
- 你生活中哪些场景适合用 RAG？（提示：企业知识库、客服FAQ、产品文档……）

---

## 活动二：用 Embedding 做语义检索

### 活动目标
上面的关键词检索很简单但不够「聪明」——「客服电话」和「联系方式」语义相近但关键词不同。
我们使用 OpenAI 的 Embedding（嵌入）接口，把文本变成「语义向量」，实现更智能的检索。

In [ ]:
import numpy as np
# 活动二：语义检索

# 不是每家服务商都提供 Embedding 接口：DeepSeek、OpenRouter 目前就没有。
# 没有的时候退化成本地「字符袋」向量，代码照样跑通，但语义能力明显变弱。
# 这本身就是一课：Embedding 的语义能力来自模型训练，不是来自余弦公式。
import zlib

USE_REAL_EMBEDDING = EMBEDDING_MODEL is not None
if not USE_REAL_EMBEDDING:
    print(f'提示：{PROVIDER} 不提供 Embedding 接口，本活动改用本地字符袋向量演示。')
    print('     想体验真正的语义检索，请把 PROVIDER 换成 openai（或本地 ollama + nomic-embed-text）。')


def local_embedding(text, dim=512):
    """本地兜底向量：把切好的词哈希到固定长度的桶里计数（词袋模型）"""
    vec = [0.0] * dim
    for word in tokenize(text):  # tokenize 来自活动一
        vec[zlib.crc32(word.encode()) % dim] += 1.0
    return vec


def get_embedding(text):
    """获取文本的向量表示：优先用服务商的 Embedding API，没有就本地兜底"""
    if not USE_REAL_EMBEDDING:
        return local_embedding(text)
    r = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return r.data[0].embedding

def cosine_similarity(a, b):
    """计算两个向量的余弦相似度"""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 为知识库中的每篇文档计算 embedding
print('正在计算文档的语义向量...')
for doc in knowledge_base:
    doc['embedding'] = get_embedding(doc['content'])
print('向量计算完成！')

# 语义检索
query = '怎么联系你们公司？'
query_emb = get_embedding(query)

# 计算相似度并排序
scored = []
for doc in knowledge_base:
    sim = cosine_similarity(query_emb, doc['embedding'])
    scored.append((sim, doc))
scored.sort(key=lambda x: x[0], reverse=True)

print(f'查询：「{query}」')
print('\n语义检索结果：')
for sim, doc in scored[:3]:
    print(f'  相似度 {sim:.3f} - {doc["title"]}')

# 用最相关的文档回答
best_doc = scored[0][1]
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'只根据提供的资料回答问题。'},
        {'role':'user','content':f'资料：{best_doc["content"]}\n\n问题：{query}'}
    ],
    temperature=0)
print(f'\nAI回答：{r.choices[0].message.content}')

### 讨论
- 语义检索和关键词检索有什么区别？各有什么适用场景？
- 「怎么联系你们公司」和知识库中的「联系方式」文档成功匹配了吗？
- Embedding 是 RAG 的核心技术之一——你现在理解它的作用了吗？

---

## 活动三：体验「智能体」——多步骤任务

### 活动目标
智能体（Agent）不仅能回答问题，还能拆解任务、分步执行。
我们通过代码模拟一个简单的智能体行为：先思考、再行动、再总结。

In [ ]:
# 活动三：简单智能体模拟

# 模拟一个「研究助手」智能体
task = '帮我查一下GPT-4o和GPT-4o-mini的区别，然后给我一个选型建议。'

# 步骤1：智能体分析任务
print('=== 步骤1：智能体分析任务 ===')
step1 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'你是一个研究助手智能体。分析用户的任务，列出需要执行的子步骤。'},
        {'role':'user','content':f'任务：{task}\n请列出完成这个任务需要的子步骤。'}
    ],
    temperature=0.3)
plan = step1.choices[0].message.content
print(plan)

# 步骤2：智能体执行研究
print('\n=== 步骤2：智能体执行研究 ===')
step2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'你是一个研究助手。基于你的知识，提供GPT-4o和GPT-4o-mini的详细对比，包括能力、价格、速度、适用场景。'},
        {'role':'user','content':'请对比GPT-4o和GPT-4o-mini'}
    ],
    temperature=0.3)
research = step2.choices[0].message.content
print(research[:400] + '...')

# 步骤3：智能体给出选型建议
print('\n=== 步骤3：智能体给出选型建议 ===')
step3 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'你是一个研究助手。基于研究结果，给出清晰、实用的建议。'},
        {'role':'user','content':f'研究结果：{research}\n\n请基于以上研究，给出选型建议。'}
    ],
    temperature=0.3)
print(step3.choices[0].message.content)

print('\n观察：智能体如何拆解任务、分步执行？')

### 讨论
- 智能体把一个大任务拆成几个子步骤？这样做好在哪？
- 智能体和普通聊天机器人最大的区别是什么？
- 实际生产中的智能体还会调用外部工具（搜索引擎/计算器/API）——你能想象一个场景吗？

---

## 本节回顾

| 技能 | 说明 |
|------|------|
| RAG 系统 | 从零搭建检索增强生成的问答系统 |
| 关键词检索 | 简单的关键词匹配检索 |
| 语义检索 | 使用 Embedding 做智能语义匹配 |
| 智能体 | 理解智能体如何拆解任务、分步执行 |

### 课后练习
1. 修改知识库内容，加入你自己的文档，测试 RAG 系统
2. 访问 perplexity.ai 体验商业级的 RAG 搜索引擎
3. 尝试 Google NotebookLM，上传一份 PDF 文档，体验个人知识库 RAG